# Phase 01: Prototype OCR Extraction

**Plan:** 01-01 — Prototyping OCR extraction and pdf2image loading

This notebook prototypes the end-to-end PDF → Image → OCR → Structured Text pipeline
on a small sample (3 PDFs) from Uthai Thani Constituency 2, running all three engines
side-by-side before scaling up in Phase 2.

**Target form:** ส.ส.5/18 (election day voting results per polling station)

**OCR Engines Under Test (in priority order per EXTR-04):**
| Priority | Engine | Mode |
|----------|--------|------|
| 1 | Typhoon OCR | API via `typhoon-ocr` package + `TYPHOON_OCR_API_KEY` |
| 2 | PaddleOCR | Local, `lang='th'` |
| 3 | Tesseract | Local, `lang=tha+eng` via `pytesseract` |

**PDF rendering:** PyMuPDF (fitz) at 200 DPI — no poppler dependency required.

## Cell 1: Environment Imports

In [3]:
import os
import re
import io
from pathlib import Path

import fitz  # PyMuPDF — PDF to image conversion
from PIL import Image
import pytesseract
from paddleocr import PaddleOCR
from typhoon_ocr import ocr_document

# Load .env for API keys
try:
    from dotenv import load_dotenv
    load_dotenv()
    print('dotenv loaded')
except ImportError:
    print('python-dotenv not installed; skipping .env load')

print(f'PyMuPDF version: {fitz.version}')
print(f'Tesseract version: {pytesseract.get_tesseract_version()}')
print('All imports OK')

/home/chatrin/Documents/Chat/CU/Year-3/2110446_DSDE_2025s2/final-project/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Checking connectivity to the model hosters, this may take a while. To bypass this check, set `PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK` to `True`.


dotenv loaded
PyMuPDF version: ('1.27.2.2', '1.27.2', None)
Tesseract version: 5.3.4
All imports OK


## Cell 2: Configuration

In [4]:
# Root directory of source PDFs (relative to notebook location)
NOTEBOOK_DIR = Path().resolve()
SOURCE_DIR = Path(os.getenv('SOURCE_DIR', str(NOTEBOOK_DIR.parent / 'source' / '\u0e40\u0e02\u0e15\u0e40\u0e25\u0e37\u0e2d\u0e01\u0e15\u0e31\u0e49\u0e07\u0e17\u0e35\u0e48 2')))

print(f'SOURCE_DIR: {SOURCE_DIR}')
print(f'SOURCE_DIR exists: {SOURCE_DIR.exists()}')

# OCR settings
PDF_DPI = 200       # Render resolution for PDF pages
SAMPLE_SIZE = 3     # Number of PDFs to process in prototype
TARGET_FORM = '\u0e2a.\u0e2a.5\u0e17\u0e31\u0e1a18.pdf'  # ส.ส.5ทับ18.pdf

# Typhoon OCR settings (reads TYPHOON_OCR_API_KEY from env)
TYPHOON_API_KEY = os.getenv('TYPHOON_OCR_API_KEY')
TYPHOON_BASE_URL = os.getenv('TYPHOON_OCR_BASE_URL', 'https://api.opentyphoon.ai/v1')
print(f'Typhoon API key set: {bool(TYPHOON_API_KEY)}')

print(f'Target form: {TARGET_FORM}')
print(f'DPI: {PDF_DPI}, Sample size: {SAMPLE_SIZE}')

SOURCE_DIR: /home/chatrin/Documents/Chat/CU/Year-3/2110446_DSDE_2025s2/final-project/source/เขตเลือกตั้งที่ 2
SOURCE_DIR exists: True
Typhoon API key set: True
Target form: ส.ส.5ทับ18.pdf
DPI: 200, Sample size: 3


## Cell 3: OCR Engine Initialization

In [ ]:
# --- PaddleOCR (PP-OCRv5, local) ---
print('Initializing PaddleOCR (PP-OCRv5)...')
paddle_ocr = PaddleOCR(
    use_doc_orientation_classify=False,
    use_doc_unwarping=False,
    use_textline_orientation=False,
)
print('PaddleOCR initialized')

# --- Tesseract (local) ---
print(f'Tesseract: {pytesseract.get_tesseract_version()}')
langs = pytesseract.get_languages()
has_thai = 'tha' in langs
print(f'Tesseract Thai lang data installed: {has_thai}')
if not has_thai:
    print('  Install with: sudo apt install tesseract-ocr-tha')

# --- Typhoon OCR (API) ---
if TYPHOON_API_KEY:
    print('Typhoon OCR API key: set')
else:
    print('WARNING: TYPHOON_OCR_API_KEY not set — Typhoon OCR cells will be skipped')

---
## Task 2: PDF Directory Traversal

In [6]:
def find_election_day_pdfs(root_dir: Path, target_filename: str = TARGET_FORM) -> list[dict]:
    """
    Recursively locate election-day PDF forms under the constituency directory.

    Expected structure:
      root_dir/
        {district}/
          {subdistrict}/
            {station}/
              ส.ส.5ทับ18.pdf

    Returns list of dicts: path, district, subdistrict, station
    """
    results = []
    for dirpath, _, filenames in os.walk(root_dir):
        if target_filename in filenames:
            dirpath = Path(dirpath)
            rel_parts = dirpath.relative_to(root_dir).parts
            results.append({
                'path': str(dirpath / target_filename),
                'district':    rel_parts[0] if len(rel_parts) > 0 else 'unknown',
                'subdistrict': rel_parts[1] if len(rel_parts) > 1 else 'unknown',
                'station':     rel_parts[2] if len(rel_parts) > 2 else 'unknown',
            })
    results.sort(key=lambda x: (x['district'], x['subdistrict'], x['station']))
    return results


pdf_list = find_election_day_pdfs(SOURCE_DIR)
print(f'Total election-day PDFs found: {len(pdf_list)}')
print('\nFirst 3 entries:')
for entry in pdf_list[:3]:
    print(f"  [{entry['district']} / {entry['subdistrict']} / {entry['station']}]")

Total election-day PDFs found: 60

First 3 entries:
  [อำเภอบ้านไร่ / ตำบลบ้านบึง / หน่วยเลือกตั้งที่ 1]
  [อำเภอบ้านไร่ / ตำบลบ้านบึง / หน่วยเลือกตั้งที่ 2]
  [อำเภอบ้านไร่ / ตำบลบ้านบึง / หน่วยเลือกตั้งที่ 3]


---
## Task 3: PDF-to-Image Conversion (PyMuPDF)

In [7]:
def pdf_to_images(pdf_path: str, dpi: int = PDF_DPI) -> list[Image.Image]:
    """
    Convert each page of a PDF to a PIL Image using PyMuPDF.
    DPI 200 gives good OCR accuracy without excessive memory use.
    """
    images = []
    scale = dpi / 72.0
    matrix = fitz.Matrix(scale, scale)
    with fitz.open(pdf_path) as doc:
        for page in doc:
            pix = page.get_pixmap(matrix=matrix)
            images.append(Image.open(io.BytesIO(pix.tobytes('png'))))
    return images


# Verify on first PDF
test_images = pdf_to_images(pdf_list[0]['path'])
print(f'Pages: {len(test_images)}')
for i, img in enumerate(test_images):
    print(f'  Page {i+1}: size={img.size}, mode={img.mode}')

Pages: 2
  Page 1: size=(1653, 2339), mode=RGB
  Page 2: size=(1653, 2339), mode=RGB


---
## Task 4a: PaddleOCR Extraction (Priority 2)

In [9]:
import numpy as np

def run_paddle_ocr(images: list[Image.Image]) -> list[dict]:
    """
    Run PaddleOCR (PP-OCRv5) on a list of PIL Images.
    Returns per-page dicts: {page, full_text, blocks}
    """
    results = []
    for page_num, img in enumerate(images):
        img_array = np.array(img)
        raw = paddle_ocr.predict(img_array)  # returns list with one OCRResult per input

        blocks, lines = [], []
        if raw:
            res_data = raw[0].json.get('res', {})
            rec_texts  = res_data.get('rec_texts', [])
            rec_scores = res_data.get('rec_scores', [])
            rec_boxes  = res_data.get('rec_boxes', [])
            for text, score, bbox in zip(rec_texts, rec_scores, rec_boxes):
                blocks.append({'text': text, 'confidence': round(score, 4), 'bbox': bbox})
                lines.append(text)

        results.append({'page': page_num + 1, 'full_text': '\n'.join(lines), 'blocks': blocks})
    return results


# Run on sample
sample_pdfs = pdf_list[:SAMPLE_SIZE]
paddle_results = []

for i, entry in enumerate(sample_pdfs):
    print(f'\n--- PaddleOCR PDF {i+1}/{SAMPLE_SIZE}: {entry["station"]} ---')
    images = pdf_to_images(entry['path'])
    pages = run_paddle_ocr(images)
    paddle_results.append({'metadata': entry, 'pages': pages})

    print(f'Page 1 blocks (first 15):')
    for blk in pages[0]['blocks'][:15]:
        print(f'  [{blk["confidence"]:.2f}] {blk["text"]}')
    has_thai = any('\u0e00' <= c <= '\u0e7f' for c in pages[0]['full_text'])
    print(f'  Contains Thai: {has_thai}')

print('\nPaddleOCR complete.')


--- PaddleOCR PDF 1/3: หน่วยเลือกตั้งที่ 1 ---
Page 1 blocks (first 15):
  [0.91] ส.ส. ๕/๑๘
  [0.99] สีขาว
  [0.99] รายงานผลการนับคะแนนสมาชิกสภาผู้แทนราษฎรแบบแบ่งเขตเลือกตั้ง
  [0.09] ะ
  [0.96] ตามที่ได้มีพระราชกฤษฎีกาให้มีการเลือกตั้งสมาชิกสภาผู้แทนราษฎร และคณะกรรมการการเลือกตั้ง
  [0.73] ได้กำหนดให้วันที...เดือน.นพ.ศ..5a
  [0.96] เป็นวันเลือกตั้ง นั้น
  [0.95] บัดนี้คณะกรรมการประจำหน่วยเลือกตั้งได้ดำเนินการนับคะแนนสมาชิกสภาผู้แทนราษฎรแบบแบ่งเขต
  [0.85] …ตำบล/แขวง/เทศบาล..
  [0.36] hr
  [0.76] อำเภอ/เขตh
  [0.94] เขตเลือกตั้งที่.. จังหวัดอุทัยธานี เสร็จสิ้นเป็นที่เรียบร้อยแล้ว ดังนั้น จึงขอรายงานผลการนับคะแนนของหน่วย
  [0.91] เลือกตั้งดังกล่าว ดังนี้
  [0.90] ๑. จำนวนผู้มีสิทธิเลือกตั้ง
  [0.76] ๑.๑ จำนวนผู้มิสิทธิเลือกตั้งตามบัญชีรายชื่อผ้มีสิธิเลือกตังจำนวน.410. คน .ทก.
  Contains Thai: True

--- PaddleOCR PDF 2/3: หน่วยเลือกตั้งที่ 2 ---
Page 1 blocks (first 15):
  [0.94] าเขตเลอกตง (แบบแบงเขตเลอุกตง)
  [0.93] ส.ส. ๕/๑๘
  [0.98] สีขาว
  [0.96] รายงานผลการนับคะแนนสมาชิกสภาผู้แทนร

---
## Task 4b: Tesseract Extraction (Priority 3)

In [ ]:
def run_tesseract_ocr(images: list[Image.Image], lang: str = 'tha+eng') -> list[dict]:
    """
    Run Tesseract on a list of PIL Images.
    Returns per-page dicts: {page, full_text}
    """
    results = []
    for page_num, img in enumerate(images):
        text = pytesseract.image_to_string(img, lang=lang)
        results.append({'page': page_num + 1, 'full_text': text.strip()})
    return results


tesseract_results = []

for i, entry in enumerate(sample_pdfs):
    print(f'\n--- Tesseract PDF {i+1}/{SAMPLE_SIZE}: {entry["station"]} ---')
    images = pdf_to_images(entry['path'])
    pages = run_tesseract_ocr(images)
    tesseract_results.append({'metadata': entry, 'pages': pages})

    preview = pages[0]['full_text'][:500].replace('\n', ' ')
    print(f'  Page 1 preview: {preview}')
    has_thai = any('\u0e00' <= c <= '\u0e7f' for c in pages[0]['full_text'])
    print(f'  Contains Thai: {has_thai}')

print('\nTesseract complete.')

---
## Task 4c: Typhoon OCR Extraction (Priority 1)

Typhoon OCR is a Thai-specialist API model. We call it directly on the PDF file
using the `typhoon-ocr` package — no manual image conversion needed.
Requires `TYPHOON_OCR_API_KEY` in `.env`.

In [ ]:
def run_typhoon_ocr(pdf_path: str, page_num: int = 1) -> str:
    """
    Run Typhoon OCR on a single PDF page.
    Returns extracted text as markdown string.
    Requires TYPHOON_OCR_API_KEY env var.
    """
    return ocr_document(
        pdf_or_image_path=pdf_path,
        task_type='v1.5',
        page_num=page_num,
        base_url=TYPHOON_BASE_URL,
        api_key=TYPHOON_API_KEY,
        figure_language='Thai',
    )


typhoon_results = []

if not TYPHOON_API_KEY:
    print('TYPHOON_OCR_API_KEY not set — skipping Typhoon OCR.')
    print('Set it in .env and rerun this cell.')
else:
    for i, entry in enumerate(sample_pdfs):
        print(f'\n--- Typhoon OCR PDF {i+1}/{SAMPLE_SIZE}: {entry["station"]} ---')
        try:
            # Process page 1 (main vote tally page)
            text = run_typhoon_ocr(entry['path'], page_num=1)
            typhoon_results.append({'metadata': entry, 'page1_text': text})
            preview = text[:600].replace('\n', ' ')
            print(f'  Page 1 preview: {preview}')
            has_thai = any('\u0e00' <= c <= '\u0e7f' for c in text)
            print(f'  Contains Thai: {has_thai}')
        except Exception as e:
            print(f'  ERROR: {e}')
            typhoon_results.append({'metadata': entry, 'page1_text': '', 'error': str(e)})

    print(f'\nTyphoon OCR complete: {len(typhoon_results)} PDFs processed.')

---
## Task 5: Side-by-Side Comparison & Regex Extraction

Compare all three engines on the same station. The regex extractor runs on Typhoon OCR
output by default (Priority 1) and falls back to PaddleOCR if Typhoon is unavailable.

In [ ]:
THAI_DIGIT_MAP = str.maketrans('\u0e50\u0e51\u0e52\u0e53\u0e54\u0e55\u0e56\u0e57\u0e58\u0e59', '0123456789')

def thai_to_arabic(text: str) -> str:
    return text.translate(THAI_DIGIT_MAP)


def extract_station_data(ocr_text: str) -> dict:
    """
    Extract key structured fields from full-page OCR text of a \u0e2a.\u0e2a.5/18 form.
    Applies regex template matching against known keyword anchors.
    """
    converted = thai_to_arabic(ocr_text)

    result = {
        'constituency_number': None,
        'station_number': None,
        'form_type_detected': False,
        'section_labels_found': [],
        'party_names': [],
        'candidate_names': [],
    }

    # Form type: ส.ส. 5/18
    if re.search(r'\u0e2a\.\u0e2a\.\s*[5\u0e55]/[18\u0e51\u0e58]', converted, re.IGNORECASE):
        result['form_type_detected'] = True

    # Constituency number
    m = re.search(r'\u0e40\u0e02\u0e15\u0e40\u0e25\u0e37\u0e2d\u0e01\u0e15\u0e31\u0e49\u0e07\u0e17\u0e35\u0e48[^\d]*(\d+)', converted)
    if m:
        result['constituency_number'] = int(m.group(1))

    # Station number
    m = re.search(r'\u0e2b\u0e19\u0e48\u0e27\u0e22\u0e40\u0e25\u0e37\u0e2d\u0e01\u0e15\u0e31\u0e49\u0e07\u0e17\u0e35\u0e48[^\d]*(\d+)', converted)
    if m:
        result['station_number'] = int(m.group(1))

    # Section labels
    known_sections = {
        '\u0e1c\u0e39\u0e49\u0e21\u0e35\u0e2a\u0e34\u0e17\u0e18\u0e34\u0e40\u0e25\u0e37\u0e2d\u0e01\u0e15\u0e31\u0e49\u0e07': '1.1_eligible_voters',
        '\u0e1a\u0e31\u0e15\u0e23\u0e14\u0e35': '2.2.1_valid_ballots',
        '\u0e1a\u0e31\u0e15\u0e23\u0e40\u0e2a\u0e35\u0e22': '2.2.2_spoiled_ballots',
        '\u0e23\u0e27\u0e21\u0e04\u0e30\u0e41\u0e19\u0e19': '3_total_score',
        '\u0e04\u0e30\u0e41\u0e19\u0e19\u0e17\u0e35\u0e48': '3_candidate_scores',
    }
    for keyword, label in known_sections.items():
        if keyword in ocr_text:
            result['section_labels_found'].append(label)

    # Known parties
    known_parties = [
        '\u0e40\u0e1e\u0e37\u0e48\u0e2d\u0e44\u0e17\u0e22',
        '\u0e20\u0e39\u0e21\u0e34\u0e43\u0e08\u0e44\u0e17\u0e22',
        '\u0e1b\u0e23\u0e30\u0e0a\u0e32\u0e0a\u0e19',
        '\u0e1b\u0e23\u0e30\u0e0a\u0e32\u0e18\u0e34\u0e1b\u0e31\u0e15\u0e22\u0e4c',
        '\u0e01\u0e25\u0e49\u0e32\u0e18\u0e23\u0e23\u0e21',
        '\u0e44\u0e17\u0e22\u0e2a\u0e23\u0e49\u0e32\u0e07\u0e44\u0e17\u0e22',
        '\u0e23\u0e27\u0e21\u0e44\u0e17\u0e22\u0e2a\u0e23\u0e49\u0e32\u0e07\u0e0a\u0e32\u0e15\u0e34',
    ]
    result['party_names'] = [p for p in known_parties if p in ocr_text]

    # Candidate names (นาย / นางสาว / นาง prefix)
    names = re.findall(
        r'(?:\u0e19\u0e32\u0e22|\u0e19\u0e32\u0e07\u0e2a\u0e32\u0e27|\u0e19\u0e32\u0e07)[^\n]+',
        ocr_text
    )
    result['candidate_names'] = [n.strip() for n in names if len(n.strip()) > 5]

    return result


# --- Side-by-side comparison ---
print('=' * 60)
print('SIDE-BY-SIDE OCR COMPARISON (Station 1)')
print('=' * 60)

first_paddle   = paddle_results[0]['pages'][0]['full_text'] if paddle_results else ''
first_tesseract = tesseract_results[0]['pages'][0]['full_text'] if tesseract_results else ''
first_typhoon  = typhoon_results[0]['page1_text'] if typhoon_results else ''

print(f'\n[PaddleOCR - first 400 chars]')
print(first_paddle[:400])

print(f'\n[Tesseract - first 400 chars]')
print(first_tesseract[:400])

print(f'\n[Typhoon OCR - first 400 chars]')
print(first_typhoon[:400] if first_typhoon else '(not run — API key not set)')

# --- Extraction using best available source ---
best_text = first_typhoon or first_paddle
print(f'\n[Regex extraction on: {"Typhoon" if first_typhoon else "PaddleOCR"}]')
extracted = extract_station_data(best_text)
for k, v in extracted.items():
    print(f'  {k}: {v}')

---
## Prototype Findings Summary

### Engine Comparison
| Engine | Thai printed text | Handwritten numbers | Setup |
|--------|------------------|---------------------|-------|
| Typhoon OCR | Expected: excellent | Expected: good (specialist) | API key required |
| PaddleOCR | Good | Poor (garbled) | Local |
| Tesseract | Fair | Poor | Local |

### Phase 2 Plan
1. Use Typhoon OCR as primary engine for handwritten vote count fields.
2. Use PaddleOCR as fallback for structural text when API is unavailable.
3. Extract station metadata from directory path (reliable, engine-independent).
4. Scale to all 60 polling stations and export `election_structured_data.csv`.

### Requirements Addressed
- **EXTR-01**: PDF-to-image conversion implemented (PyMuPDF, 200 DPI)
- **EXTR-02**: Directory traversal finds all ส.ส.5/18 PDFs
- **EXTR-03**: Thai text extracted (PaddleOCR + Tesseract + Typhoon OCR)
- **EXTR-04**: All three engines tested in priority order (Typhoon > PaddleOCR > Tesseract)